# SciFact BM25 Retrieval

BM25 baseline on SciFact using [Pyserini](https://github.com/castorini/pyserini).
Pipeline:
1. Load SciFact corpus & queries (same `load_data` module as other notebooks)
2. Write corpus to a temp JSONL directory
3. Build a Lucene index with Pyserini
4. BM25 retrieval per query
5. Evaluate with the shared `evaluate_run` function
6. Save results and compare with Baseline (dense) & HyDE

In [1]:
# Install pyserini if not already available (requires Java 11+)
import importlib, sys
if importlib.util.find_spec('pyserini') is None:
    !{sys.executable} -m pip install pyserini
else:
    print('pyserini already installed')

pyserini already installed


In [2]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['OMP_NUM_THREADS'] = '1'
print('Thread guards enabled.')

Thread guards enabled.


In [3]:
import json
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
from tqdm import tqdm

from config import DATASET_SPLIT, PREFER_BEIR, MAX_QUERIES, TOP_KS
from load_data import load_scifact_data
from evaluate import evaluate_run

print('Imports OK')

/Users/winstondong/miniforge3/envs/adnlp_3.2_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


## Load SciFact Data

In [4]:
SPLIT           = DATASET_SPLIT
PREFER_BEIR_LOCAL = PREFER_BEIR
MAX_QUERIES_LOCAL = MAX_QUERIES
TOP_KS_LOCAL    = TOP_KS
MAX_K           = max(TOP_KS_LOCAL)
OUTPUT_DIR      = Path('results/bm25')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

corpus, queries, qrels, data_source = load_scifact_data(
    split=SPLIT,
    prefer_beir=PREFER_BEIR_LOCAL,
    max_queries=MAX_QUERIES_LOCAL,
)

query_ids      = list(queries.keys())
corpus_doc_ids = list(corpus.keys())

print(f'Data source : {data_source}')
print(f'Corpus size : {len(corpus_doc_ids)}')
print(f'Queries     : {len(query_ids)}')

Data source : mteb/scifact (corpus:corpus/corpus, queries:queries/queries, qrels:default/test)
Corpus size : 5183
Queries     : 300


## Build Lucene BM25 Index

Pyserini's `JsonCollection` expects one JSONL file (or multiple) where each line has at minimum an `id` and a `contents` field.  
We write the SciFact corpus to a temporary directory, build the index, then clean up the raw files.

In [5]:
# Write corpus to a temp JSONL file for Pyserini indexing
tmp_corpus_dir = Path(tempfile.mkdtemp(prefix='scifact_corpus_'))
jsonl_path = tmp_corpus_dir / 'corpus.jsonl'

with jsonl_path.open('w', encoding='utf-8') as f:
    for did in corpus_doc_ids:
        record = {'id': str(did), 'contents': corpus[did]}
        f.write(json.dumps(record, ensure_ascii=False) + '\n')

print(f'Wrote {len(corpus_doc_ids)} docs to {jsonl_path}')

Wrote 5183 docs to /var/folders/kp/m3wqzg2d66v1h4x669hl3vhc0000gn/T/scifact_corpus_km542nwy/corpus.jsonl


In [6]:
index_dir = Path('bm25_scifact_index')
if index_dir.exists():
    shutil.rmtree(index_dir)

cmd = [
    sys.executable, '-m', 'pyserini.index.lucene',
    '--collection', 'JsonCollection',
    '--input',  str(tmp_corpus_dir),
    '--index',  str(index_dir),
    '--generator', 'DefaultLuceneDocumentGenerator',
    '--threads', '1',
    '--storePositions',
    '--storeDocvectors',
    '--storeRaw',
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-3000:] if result.stdout else '')
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])
    raise RuntimeError('Indexing failed')
print(f'Index built at: {index_dir}')

# Clean up temp corpus files
shutil.rmtree(tmp_corpus_dir)

Running: /Users/winstondong/miniforge3/envs/adnlp_3.2_env/bin/python -m pyserini.index.lucene --collection JsonCollection --input /var/folders/kp/m3wqzg2d66v1h4x669hl3vhc0000gn/T/scifact_corpus_km542nwy --index bm25_scifact_index --generator DefaultLuceneDocumentGenerator --threads 1 --storePositions --storeDocvectors --storeRaw
-01 16:29:46,615 INFO  [main] index.IndexCollection (IndexCollection.java:189) -  + Generator: DefaultLuceneDocumentGenerator
2026-04-01 16:29:46,615 INFO  [main] index.IndexCollection (IndexCollection.java:190) -  + Language: en
2026-04-01 16:29:46,615 INFO  [main] index.IndexCollection (IndexCollection.java:191) -  + Stemmer: porter
2026-04-01 16:29:46,615 INFO  [main] index.IndexCollection (IndexCollection.java:192) -  + Keep stopwords? false
2026-04-01 16:29:46,616 INFO  [main] index.IndexCollection (IndexCollection.java:193) -  + Stopwords: null
2026-04-01 16:29:46,616 INFO  [main] index.IndexCollection (IndexCollection.java:194) -  + Store positions? true

## BM25 Retrieval

In [7]:
from pyserini.search.lucene import LuceneSearcher

searcher = LuceneSearcher(str(index_dir))
searcher.set_bm25(k1=0.9, b=0.4)  # standard BEIR-tuned BM25 params

per_query_retrieved = {}
per_query_rows      = []

for qid in tqdm(query_ids, desc='BM25 retrieval'):
    query_text = queries[qid]
    hits = searcher.search(query_text, k=MAX_K)
    retrieved_doc_ids = [hit.docid for hit in hits]
    per_query_retrieved[qid] = retrieved_doc_ids

    gold = qrels[qid]
    row = {
        'mode': 'bm25',
        'query_id': qid,
        'query': query_text,
        'gold_doc_ids': sorted(gold),
        'retrieved_doc_ids': retrieved_doc_ids,
        'hit@1':  int(any(d in gold for d in retrieved_doc_ids[:1])),
        'hit@5':  int(any(d in gold for d in retrieved_doc_ids[:5])),
        'hit@10': int(any(d in gold for d in retrieved_doc_ids[:10])),
    }
    per_query_rows.append(row)

print(f'Retrieved results for {len(per_query_retrieved)} queries')

Apr 01, 2026 4:29:51 PM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false
BM25 retrieval: 100%|██████████| 300/300 [00:00<00:00, 426.97it/s]

Retrieved results for 300 queries


## Evaluate

In [8]:
metrics = evaluate_run(per_query_retrieved, qrels, TOP_KS_LOCAL)

print('=== BM25 Results ===')
for k in ['Recall@1', 'Recall@5', 'Recall@10', 'MRR@10', 'nDCG@10']:
    if k in metrics:
        print(f'{k:<12}: {metrics[k]:.4f}')

=== BM25 Results ===
Recall@1    : 0.5371
Recall@10   : 0.8072
MRR@10      : 0.6460
nDCG@10     : 0.6799


In [9]:
metrics_payload = {
    'config': {
        'mode': 'bm25',
        'split': SPLIT,
        'prefer_beir': PREFER_BEIR_LOCAL,
        'data_source': data_source,
        'bm25_k1': 0.9,
        'bm25_b': 0.4,
        'top_ks': TOP_KS_LOCAL,
        'max_queries': MAX_QUERIES_LOCAL,
    },
    'bm25': metrics,
}

metrics_path = OUTPUT_DIR / 'metrics.json'
with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)

per_query_path = OUTPUT_DIR / 'per_query_results.jsonl'
with per_query_path.open('w', encoding='utf-8') as f:
    for row in per_query_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'Saved metrics      : {metrics_path}')
print(f'Saved per-query    : {per_query_path}')

Saved metrics      : results/bm25/metrics.json
Saved per-query    : results/bm25/per_query_results.jsonl


## Comparison: BM25 vs Baseline (Dense) vs HyDE

Reads the saved `metrics.json` from each experiment folder and prints a side-by-side table.

In [10]:
def load_metrics(path: Path, key: str):
    """Load a metrics dict from a saved metrics.json file."""
    if not path.exists():
        return None
    with path.open() as f:
        data = json.load(f)
    return data.get(key)

results_root = Path('results')
experiments = [
    ('BM25',          load_metrics(results_root / 'bm25'      / 'metrics.json', 'bm25')),
    ('Baseline',      load_metrics(results_root / 'baseline'  / 'metrics.json', 'baseline')),
    ('HyDE',          load_metrics(results_root / 'hyde'      / 'metrics.json', 'hyde')),
]

metric_keys = ['Recall@1', 'Recall@5', 'Recall@10', 'MRR@10', 'nDCG@10']

col_w = 12
header = f"{'Method':<14}" + ''.join(f'{k:>{col_w}}' for k in metric_keys)
print(header)
print('-' * len(header))
for name, m in experiments:
    if m is None:
        print(f'{name:<14}' + ' ' * col_w * len(metric_keys) + '  (results not found)')
    else:
        row = f'{name:<14}' + ''.join(f'{m.get(k, float("nan")):>{col_w}.4f}' for k in metric_keys)
        print(row)

Method            Recall@1    Recall@5   Recall@10      MRR@10     nDCG@10
--------------------------------------------------------------------------
BM25                0.5371         nan      0.8072      0.6460      0.6799
Baseline            0.5757      0.8035      0.8566      0.6936      0.7296
HyDE                0.5683      0.8109      0.8824      0.6901      0.7338
